# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR² dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {metadata.cite_as}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We'll list all available record sets, their fields, and field `@id`s.

In [ ]:
# List all record sets
record_sets = list(dataset.record_sets)
print(f"Record sets (@id): {[rs['@id'] for rs in record_sets]}")
print()
for rs in record_sets:
    print(f"Record set name: {rs['name']}")
    print(f"  @id: {rs['@id']}")
    print(f"  Description: {rs.get('description','N/A')}")
    if 'field' in rs:
        fields = rs['field']
        # In Croissant, field could be an OrderedDict (single) or list
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            print(f"    - {field['name']} (@id: {field['@id']}) | {field.get('description', '')}")
    print()

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Replace these IDs with the ones shown in the overview step
# Here, we programmatically collect and load all available record sets
dataframes = dict()
loaded_record_sets = []
for rs in dataset.record_sets:
    rs_id = rs['@id']
    loaded_record_sets.append(rs_id)
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {rs_id}")
    else:
        print(f"No records found for record set {rs_id}")
print()
if dataframes:
    # Pick the first loaded record set for sample output:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Columns in {main_rs_id}: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Please adjust these field IDs and record set IDs according to step 2 output
record_set_id = main_rs_id  # Already selected from above
df = dataframes[record_set_id].copy()
print(f"DataFrame shape: {df.shape}")

# Try to select a plausible numeric field (e.g. molecular weight, coverage, etc)
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in {'i','f'}]
if not numeric_candidates:
    numeric_candidates = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notna().any()]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")
    # Ensure the column is numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Look for any categorical/grouping fields
    group_candidates = [col for col in df.columns if (df[col].dtype=='object') & (df[col].nunique() > 1) & (df[col].nunique() < df.shape[0]//2)]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric fields detected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Visualize the numeric field distribution
if numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If grouped, show mean per group
    if group_candidates:
        plt.figure(figsize=(12,4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR² dataset schema using `mlcroissant`, listed and examined available record sets and their fields by `@id`, loaded and visualized key data, and performed initial data filtering and normalization. You are now equipped to further analyze protein abundance, modifications, or other attributes depending on your research goals.

For further analysis, consider leveraging the field `@id`s to subset specific variables, or link annotations for cross-referencing with biological knowledge bases such as UniProt.